# HISTORICAL CUSTOMER LIFETIME VALUE

# Step 1: Import libraries and load dataset

In [14]:
import pandas as pd
import numpy as np
import os
print("Libraries imported succesfully")

df = pd.read_csv(r"C:\Users\onyer\Downloads\cleaned_retail_data.csv")
print(f"\nDataset shape: {df.shape}")
print(f"\nColums: {df.columns.tolist()}")
df.head()

Libraries imported succesfully

Dataset shape: (392692, 10)

Colums: ['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country', 'TransactionMonth', 'CohortMonth']


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TransactionMonth,CohortMonth
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom,2010-12,2010-12
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,2010-12,2010-12
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom,2010-12,2010-12
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,2010-12,2010-12
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,2010-12,2010-12


# Step 2: Calculate TotalRevenue per transaction

In [5]:
df["TotalRevenue"] = df["Quantity"] * df["UnitPrice"]

print("TotalRevenue Column added successfully")
print(df[["CustomerID", "InvoiceNo", "Quantity", "UnitPrice", "TotalRevenue"]])

TotalRevenue Column added successfully
        CustomerID  InvoiceNo  Quantity  UnitPrice  TotalRevenue
0            17850     536365         6       2.55         15.30
1            17850     536365         6       3.39         20.34
2            17850     536365         8       2.75         22.00
3            17850     536365         6       3.39         20.34
4            17850     536365         6       3.39         20.34
...            ...        ...       ...        ...           ...
392687       12680     581587        12       0.85         10.20
392688       12680     581587         6       2.10         12.60
392689       12680     581587         4       4.15         16.60
392690       12680     581587         4       4.15         16.60
392691       12680     581587         3       4.95         14.85

[392692 rows x 5 columns]


# Step 3: AOV

In [7]:
try:
    aov = pd.read_csv('outputs/siva_aov.csv')
    print("✅ siva AOV output loaded successfully")

except FileNotFoundError:
    print("⚠️ siva output not found - using fallback AOV calculation")
    
    customer_revenue = df.groupby('CustomerID')['TotalRevenue'].sum()
    customer_orders = df.groupby('CustomerID')['InvoiceNo'].nunique()
    aov = (customer_revenue / customer_orders).reset_index()
    aov.columns = ['CustomerID', 'AOV']

print(aov.head())

⚠️ siva output not found - using fallback AOV calculation
   CustomerID           AOV
0       12346  77183.600000
1       12347    615.714286
2       12348    449.310000
3       12349   1757.550000
4       12350    334.400000


# Step 4: Purchase frequency

In [11]:
try:
    purchase_freq = pd.read_csv('outputs/yash_frequency.csv')
    print("✅ yash output loaded successfully")

except FileNotFoundError:
    print("⚠️ yash output not found - using fallback")
    
    purchase_freq = df.groupby('CustomerID')['InvoiceNo'].nunique().reset_index()
    purchase_freq.columns = ['CustomerID', 'PurchaseFrequency']

print(purchase_freq.head())

⚠️ yash output not found - using fallback
   CustomerID  PurchaseFrequency
0       12346                  1
1       12347                  7
2       12348                  4
3       12349                  1
4       12350                  1


# Step 5: Historical CLTV

In [12]:
# Merge AOV and Purchase Frequency
cltv_df = aov.merge(purchase_freq, on='CustomerID')

# CLTV = AOV x Purchase Frequency
cltv_df['CLTV'] = cltv_df['AOV'] * cltv_df['PurchaseFrequency']

print(cltv_df.head(10))
print(f"\nCLTV Summary:\n{cltv_df['CLTV'].describe()}")

   CustomerID           AOV  PurchaseFrequency      CLTV
0       12346  77183.600000                  1  77183.60
1       12347    615.714286                  7   4310.00
2       12348    449.310000                  4   1797.24
3       12349   1757.550000                  1   1757.55
4       12350    334.400000                  1    334.40
5       12352    313.255000                  8   2506.04
6       12353     89.000000                  1     89.00
7       12354   1079.400000                  1   1079.40
8       12355    459.400000                  1    459.40
9       12356    937.143333                  3   2811.43

CLTV Summary:
count      4338.000000
mean       2048.688081
std        8985.230220
min           3.750000
25%         306.482500
50%         668.570000
75%        1660.597500
max      280206.020000
Name: CLTV, dtype: float64


In [15]:
os.makedirs('outputs', exist_ok=True)
cltv_df.to_csv('outputs/hiren_cltv.csv', index=False)
print("Saved successfully")

Saved successfully
